In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time
import random

# =======================
# Config
# =======================
CITY_SOURCE = "list"   # "list" or "csv"
CITIES_CSV_PATH = r"C:\path\to\cities.csv"  # used only if CITY_SOURCE="csv", must have a 'city' column

# Edit/extend this list freely if CITY_SOURCE="list"
CITIES = [
    "mumbai", "delhi", "ahmedabad",
    "bengaluru", "hyderabad", "pune",
    "jaipur", "chennai", "kolkata",
    "goa", "udaipur", "varanasi", "surat"
]

MONTHS = ["2025-11", "2025-12", "2026-01"]  # Nov, Dec, Jan
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
RESULT_CSV = "booking_hotels.csv"  # unified name

# Adults/rooms (adjust if you want)
ADULTS = 2
ROOMS = 1
CURRENCY = "INR"

# politeness/safety
BASE_DELAY = (1.5, 3.0)  # sleep seconds range between requests
MAX_RETRIES = 3

# =======================
# Input: stay duration
# =======================
try:
    STAY_DURATION = int(input("Enter number of nights to stay: ").strip())
    if STAY_DURATION <= 0:
        raise ValueError
except (ValueError, EOFError):
    print("Invalid or no input. Defaulting to 2 nights.")
    STAY_DURATION = 2

# =======================
# Helpers
# =======================
def generate_dates(month: str):
    """Return a list of check-in dates spaced every 5 days within the month."""
    start_date = datetime.strptime(month + "-01", "%Y-%m-%d")
    next_month = (start_date.replace(day=28) + timedelta(days=4)).replace(day=1)
    days_in_month = (next_month - start_date).days
    return [start_date + timedelta(days=i) for i in range(0, days_in_month, 5)]

def get_cities():
    if CITY_SOURCE.lower() == "csv":
        dfc = pd.read_csv(CITIES_CSV_PATH)
        if "city" not in dfc.columns:
            raise ValueError(f"CSV at {CITIES_CSV_PATH} must have a 'city' column.")
        # de-dup, dropna, strip
        cities = (
            dfc["city"].dropna().astype(str).str.strip().str.lower().unique().tolist()
        )
        return [c for c in cities if c]
    return [c.strip().lower() for c in CITIES if c.strip()]

def booking_search_url(city: str, checkin: datetime, checkout: datetime, rows: int = 25):
    # Build URL via params to keep it neat
    params = {
        "ss": city,
        "checkin_year": checkin.year,
        "checkin_month": checkin.month,
        "checkin_monthday": checkin.day,
        "checkout_year": checkout.year,
        "checkout_month": checkout.month,
        "checkout_monthday": checkout.day,
        "group_adults": ADULTS,
        "no_rooms": ROOMS,
        "rows": rows,
        "selected_currency": CURRENCY,
        # you can add more filters here (price range, star rating, etc.)
    }
    base = "https://www.booking.com/searchresults.html"
    # Turn params into querystring
    return base, params

def fetch_html(url: str, params: dict, headers: dict, max_retries: int = 3):
    """Simple retry with backoff."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=25)
            if resp.status_code == 200:
                return resp.text
            # handle mild throttling
            if resp.status_code in (429, 503):
                sleep_s = 2 * attempt + random.uniform(0.2, 0.8)
                time.sleep(sleep_s)
            else:
                # other non-200, break or keep trying
                time.sleep(1 + attempt * 0.5)
        except requests.RequestException:
            time.sleep(1 + attempt * 0.5)
    raise RuntimeError(f"Failed to fetch after {max_retries} attempts.")

def scrape_booking(city: str, checkin: datetime, checkout: datetime):
    url, params = booking_search_url(city, checkin, checkout)
    html = fetch_html(url, params, HEADERS, MAX_RETRIES)
    soup = BeautifulSoup(html, "html.parser")

    hotels = []
    listings = soup.select("div[data-testid='property-card']")
    for item in listings:
        name = item.select_one("div[data-testid='title']")
        name = name.get_text(strip=True) if name else None

        price = item.select_one("span[data-testid='price-and-discounted-price']")
        price = price.get_text(strip=True) if price else None

        rating_box = item.select_one("div[data-testid='review-score']")
        rating = None
        if rating_box:
            # Usually the numeric score appears first line
            rating = rating_box.get_text("\n", strip=True).split("\n")[0]

        location = item.select_one("span[data-testid='address']")
        location = location.get_text(strip=True) if location else None

        link = item.select_one("a[data-testid='title-link']")
        link = ("https://www.booking.com" + link["href"]) if link and link.has_attr("href") else None

        hotels.append({
            "City": city.title(),
            "Check_in": checkin.strftime("%Y-%m-%d"),
            "Check_out": checkout.strftime("%Y-%m-%d"),
            "Hotel_Name": name,
            "Location": location,
            "Rating": rating,
            "Price": price,
            "URL": link
        })
    return hotels

# =======================
# Main
# =======================
all_rows = []
cities = get_cities()
for city in cities:
    for month in MONTHS:
        for checkin in generate_dates(month):
            checkout = checkin + timedelta(days=STAY_DURATION)
            print(f"Scraping {city.title()}  {checkin.date()} → {checkout.date()} ...")
            try:
                batch = scrape_booking(city, checkin, checkout)
                all_rows.extend(batch)
            except Exception as e:
                print(f"  ⚠️  Error: {e}")
            # polite random delay to reduce blocks
            time.sleep(random.uniform(*BASE_DELAY))

# =======================
# Save
# =======================
df = pd.DataFrame(all_rows)

# drop obvious empties
if not df.empty:
    # de-dup by (City, Hotel_Name, Check_in, Check_out)
    df = df.drop_duplicates(subset=["City", "Hotel_Name", "Check_in", "Check_out"])
    # optional: keep only non-empty names/links
    df = df[(df["Hotel_Name"].notna()) & (df["URL"].notna())]

df.to_csv(RESULT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✅ Data saved to {RESULT_CSV} ({len(df)} rows)")


Enter number of nights to stay:  3


Scraping Mumbai  2025-11-01 → 2025-11-04 ...
Scraping Mumbai  2025-11-06 → 2025-11-09 ...
Scraping Mumbai  2025-11-11 → 2025-11-14 ...
Scraping Mumbai  2025-11-16 → 2025-11-19 ...
Scraping Mumbai  2025-11-21 → 2025-11-24 ...
Scraping Mumbai  2025-11-26 → 2025-11-29 ...
Scraping Mumbai  2025-12-01 → 2025-12-04 ...
Scraping Mumbai  2025-12-06 → 2025-12-09 ...
Scraping Mumbai  2025-12-11 → 2025-12-14 ...
Scraping Mumbai  2025-12-16 → 2025-12-19 ...
Scraping Mumbai  2025-12-21 → 2025-12-24 ...
Scraping Mumbai  2025-12-26 → 2025-12-29 ...
Scraping Mumbai  2025-12-31 → 2026-01-03 ...
Scraping Mumbai  2026-01-01 → 2026-01-04 ...
  ⚠️  Error: Failed to fetch after 3 attempts.
Scraping Mumbai  2026-01-06 → 2026-01-09 ...
Scraping Mumbai  2026-01-11 → 2026-01-14 ...
Scraping Mumbai  2026-01-16 → 2026-01-19 ...
Scraping Mumbai  2026-01-21 → 2026-01-24 ...
Scraping Mumbai  2026-01-26 → 2026-01-29 ...
Scraping Mumbai  2026-01-31 → 2026-02-03 ...
Scraping Delhi  2025-11-01 → 2025-11-04 ...
Scraping 